# MCP RAG Server

This notebook creates a FastMCP server that exposes RAG functionality as tools for Claude.

## Architecture

```
┌─────────────────┐     ┌─────────────────┐     ┌─────────────────┐
│   Claude API    │────▶│   MCP Server    │────▶│   RAG Index     │
│   (or Claude    │     │   (FastMCP)     │     │   (FAISS+BM25)  │
│    Desktop)     │◀────│                 │◀────│                 │
└─────────────────┘     └─────────────────┘     └─────────────────┘
```

## Tools Exposed

1. `search_documents` - Search the knowledge base
2. `get_context` - Get formatted context for a query
3. `list_sources` - List available source files
4. `get_document_info` - Get info about a specific document

In [152]:
# Install FastMCP and dependencies
!pip install fastmcp anthropic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.3/413.3 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 388.2/388.2 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.0/244.0 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.4/197.4 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.1/233.1 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.3/96.3 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.4/67.4 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.4/243.4 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.5/68.5 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [153]:
import json
import pickle
from pathlib import Path
from dataclasses import dataclass
from typing import Optional
import numpy as np

# These will be loaded from the saved indexes
from sentence_transformers import SentenceTransformer
import faiss
from rank_bm25 import BM25Okapi
import string

## Load RAG Components

First, we need to recreate the RAG classes and load the saved indexes.

In [154]:
@dataclass
class ChunkWithMetadata:
    """A text chunk with associated metadata for RAG retrieval."""
    text: str
    source_file: str = ""
    source_path: str = ""
    element_type: str = ""
    chunk_index: int = 0
    total_chunks: int = 0
    page_number: Optional[int] = None
    section: str = ""
    char_start: int = 0
    char_end: int = 0
    
    def to_dict(self) -> dict:
        return {
            'text': self.text,
            'source_file': self.source_file,
            'source_path': self.source_path,
            'element_type': self.element_type,
            'chunk_index': self.chunk_index,
            'total_chunks': self.total_chunks,
            'page_number': self.page_number,
            'section': self.section,
            'char_start': self.char_start,
            'char_end': self.char_end
        }
    
    @classmethod
    def from_dict(cls, d: dict) -> 'ChunkWithMetadata':
        return cls(**d)
    
    def citation(self) -> str:
        parts = []
        if self.source_file:
            parts.append(self.source_file)
        if self.page_number:
            parts.append(f"p.{self.page_number}")
        if self.section:
            parts.append(f"§{self.section}")
        return " | ".join(parts) if parts else "Unknown source"

In [155]:
class QueryExpander:
    """Expands queries to improve retrieval recall."""
    
    SYNONYMS = {
        'ai': ['artificial intelligence', 'machine learning', 'ml', 'deep learning'],
        'ml': ['machine learning', 'ai', 'artificial intelligence'],
        'nlp': ['natural language processing', 'text processing', 'language models'],
        'llm': ['large language model', 'language model', 'gpt', 'transformer'],
        'research': ['study', 'investigation', 'analysis', 'paper'],
        'paper': ['publication', 'article', 'study', 'research'],
        'model': ['algorithm', 'system', 'method', 'approach'],
        'workshop': ['seminar', 'conference', 'meeting', 'session'],
        'collaboration': ['partnership', 'cooperation', 'teamwork'],
        'grant': ['funding', 'award', 'proposal'],
    }
    
    ACRONYMS = {
        'ai': 'artificial intelligence',
        'ml': 'machine learning',
        'nlp': 'natural language processing',
        'llm': 'large language model',
        'nsf': 'national science foundation',
        'nih': 'national institutes of health',
    }
    
    def expand_query(self, query: str) -> list[str]:
        queries = [query]
        query_lower = query.lower()
        words = query_lower.split()
        
        # Expand acronyms
        expanded = query_lower
        for acronym, full_form in self.ACRONYMS.items():
            if acronym in words:
                expanded = expanded.replace(acronym, full_form)
        if expanded != query_lower:
            queries.append(expanded)
        
        # Add synonym variations
        for word in words:
            if word in self.SYNONYMS:
                for synonym in self.SYNONYMS[word][:2]:
                    new_query = query_lower.replace(word, synonym)
                    if new_query not in queries:
                        queries.append(new_query)
        
        return queries

In [156]:
class HybridRAGServer:
    """
    Hybrid RAG server that can be used by MCP tools.
    Loads pre-built indexes from disk.
    """
    
    def __init__(self, index_path: str = "Data/rag_indexes/hybrid"):
        self.index_path = Path(index_path)
        self.model = None
        self.faiss_index = None
        self.bm25_index = None
        self.chunks: list[ChunkWithMetadata] = []
        self.tokenized_chunks = []
        self.expander = QueryExpander()
        self.semantic_weight = 0.7
        self.keyword_weight = 0.3
        self.model_name = "all-MiniLM-L6-v2"
        
    def load(self) -> None:
        """Load indexes from disk."""
        print(f"Loading RAG index from {self.index_path}...")
        
        # Load FAISS index
        self.faiss_index = faiss.read_index(str(self.index_path / "faiss_index.faiss"))
        
        # Load chunks and BM25 data
        with open(self.index_path / "hybrid_data.pkl", 'rb') as f:
            data = pickle.load(f)
            self.chunks = [ChunkWithMetadata.from_dict(c) for c in data['chunks']]
            self.tokenized_chunks = data['tokenized_chunks']
            self.model_name = data.get('model_name', self.model_name)
            self.semantic_weight = data.get('semantic_weight', 0.7)
            self.keyword_weight = 1 - self.semantic_weight
        
        # Rebuild BM25 index
        self.bm25_index = BM25Okapi(self.tokenized_chunks)
        
        # Load embedding model
        print(f"Loading embedding model: {self.model_name}")
        self.model = SentenceTransformer(self.model_name)
        
        print(f"Loaded {self.faiss_index.ntotal} vectors, {len(self.chunks)} chunks")
    
    def _tokenize(self, text: str) -> list[str]:
        text = text.lower().translate(str.maketrans('', '', string.punctuation))
        return [t for t in text.split() if len(t) > 2]
    
    def search(self, query: str, top_k: int = 5, expand: bool = True) -> list[dict]:
        """
        Search the knowledge base.
        
        Returns list of results with text, citation, and scores.
        """
        queries = self.expander.expand_query(query) if expand else [query]
        all_results = {}
        
        for i, q in enumerate(queries):
            weight = 1.0 if i == 0 else 0.5
            
            # Semantic search
            query_embedding = self.model.encode([q], convert_to_numpy=True)
            faiss.normalize_L2(query_embedding)
            sem_scores, sem_indices = self.faiss_index.search(query_embedding, top_k * 2)
            
            # Keyword search
            query_tokens = self._tokenize(q)
            kw_scores = self.bm25_index.get_scores(query_tokens)
            max_kw = max(kw_scores) if max(kw_scores) > 0 else 1
            
            # Combine
            for idx, sem_score in zip(sem_indices[0], sem_scores[0]):
                if idx < 0:
                    continue
                kw_score = kw_scores[idx] / max_kw
                combined = self.semantic_weight * sem_score + self.keyword_weight * kw_score
                
                if idx not in all_results:
                    all_results[idx] = {
                        'text': self.chunks[idx].text,
                        'citation': self.chunks[idx].citation(),
                        'source_file': self.chunks[idx].source_file,
                        'source_path': self.chunks[idx].source_path,
                        'score': 0,
                        'semantic_score': 0,
                        'keyword_score': 0,
                    }
                
                all_results[idx]['score'] += combined * weight
                all_results[idx]['semantic_score'] = max(all_results[idx]['semantic_score'], float(sem_score))
                all_results[idx]['keyword_score'] = max(all_results[idx]['keyword_score'], kw_score)
        
        results = sorted(all_results.values(), key=lambda x: x['score'], reverse=True)
        return results[:top_k]
    
    def get_context(self, query: str, top_k: int = 5, max_chars: int = 8000) -> tuple[str, list[str]]:
        """
        Get formatted context string for LLM prompt.
        
        Returns (context_string, list_of_citations)
        """
        results = self.search(query, top_k=top_k)
        
        context_parts = []
        citations = []
        total_chars = 0
        
        for i, r in enumerate(results):
            if total_chars + len(r['text']) > max_chars:
                break
            
            citations.append(r['citation'])
            context_parts.append(
                f"[Source {i+1}: {r['citation']}]\n{r['text']}"
            )
            total_chars += len(r['text'])
        
        return "\n\n".join(context_parts), citations
    
    def list_sources(self) -> list[dict]:
        """List all unique source files in the index."""
        sources = {}
        for chunk in self.chunks:
            if chunk.source_file not in sources:
                sources[chunk.source_file] = {
                    'file': chunk.source_file,
                    'path': chunk.source_path,
                    'chunk_count': 0
                }
            sources[chunk.source_file]['chunk_count'] += 1
        
        return sorted(sources.values(), key=lambda x: x['chunk_count'], reverse=True)


# Initialize the RAG server
rag_server = HybridRAGServer("Data/rag_indexes/hybrid")
rag_server.load()

Loading RAG index from Data/rag_indexes/hybrid...
Loading embedding model: all-MiniLM-L6-v2
Loaded 5804 vectors, 5804 chunks


In [157]:
# Test the RAG server
results = rag_server.search("AI research collaboration", top_k=3)

print("Test search results:")
for i, r in enumerate(results):
    print(f"\n[{i+1}] {r['citation']}")
    print(f"    Score: {r['score']:.3f}")
    print(f"    {r['text'][:100]}...")

Test search results:

[1] KLAB CHORUS
    Score: 2.588
    vancing the understanding of human-AI collaboration. This effort will last from year 1 until year 4....

[2] KLAB CHORUS
    Score: 2.182
    n, and examines what emerges when multiple AI agents collaborate as a miniature research laboratory....

[3] KLAB CHORUS
    Score: 1.906
    be imbued with the perspective-taking, creativity, and reasoning required for scientific discovery. ...


## FastMCP Server

Now we create the MCP server that exposes RAG functionality as tools.

In [158]:
from fastmcp import FastMCP

# Create MCP server
mcp = FastMCP("CHORUS RAG Server")


@mcp.tool()
def search_documents(query: str, top_k: int = 5) -> str:
    """
    Search the CHORUS knowledge base for relevant documents.
    
    Args:
        query: The search query (natural language)
        top_k: Number of results to return (default 5)
    
    Returns:
        JSON string with search results including text, citations, and scores
    """
    results = rag_server.search(query, top_k=top_k)
    return json.dumps(results, indent=2)


@mcp.tool()
def get_context_for_question(question: str, max_sources: int = 5) -> str:
    """
    Get relevant context from the knowledge base to answer a question.
    
    Use this to retrieve information before answering questions about:
    - Research projects and collaborations
    - Grants and funding (NSF, NIH, etc.)
    - Workshops and presentations
    - Team members and their work
    
    Args:
        question: The question to find context for
        max_sources: Maximum number of sources to include
    
    Returns:
        Formatted context with citations
    """
    context, citations = rag_server.get_context(question, top_k=max_sources)
    
    result = {
        'context': context,
        'citations': citations,
        'num_sources': len(citations)
    }
    return json.dumps(result, indent=2)


@mcp.tool()
def list_available_sources() -> str:
    """
    List all source documents available in the knowledge base.
    
    Returns:
        JSON list of sources with file names and chunk counts
    """
    sources = rag_server.list_sources()
    return json.dumps({
        'total_sources': len(sources),
        'total_chunks': sum(s['chunk_count'] for s in sources),
        'sources': sources[:50]  # Limit to top 50
    }, indent=2)


@mcp.tool()
def search_by_source(source_name: str, query: str = "", top_k: int = 5) -> str:
    """
    Search within a specific source document.
    
    Args:
        source_name: Partial or full name of the source file
        query: Optional query to filter results (if empty, returns all chunks from source)
        top_k: Number of results to return
    
    Returns:
        JSON with matching chunks from the specified source
    """
    # Find matching chunks
    matching_chunks = [
        {'text': c.text, 'citation': c.citation(), 'chunk_index': c.chunk_index}
        for c in rag_server.chunks
        if source_name.lower() in c.source_file.lower()
    ]
    
    if not matching_chunks:
        return json.dumps({'error': f'No source found matching "{source_name}"'})
    
    if query:
        # Filter by query relevance
        results = rag_server.search(query, top_k=top_k * 3)
        results = [r for r in results if source_name.lower() in r['source_file'].lower()]
        return json.dumps({'results': results[:top_k]})
    
    return json.dumps({
        'source': source_name,
        'total_chunks': len(matching_chunks),
        'chunks': matching_chunks[:top_k]
    }, indent=2)


print("MCP server created with tools:")
print("  - search_documents")
print("  - get_context_for_question")
print("  - list_available_sources")
print("  - search_by_source")

MCP server created with tools:
  - search_documents
  - get_context_for_question
  - list_available_sources
  - search_by_source


## Integration with Claude API

There are two ways to use this MCP server with Claude:

### Option 1: Direct Tool Use (Recommended for API)

Use the Anthropic API directly with tool definitions. This doesn't require MCP - you define tools as functions and Claude calls them.

In [159]:
import anthropic
import os

# Define tools for Claude API
TOOLS = [
    {
        "name": "search_documents",
        "description": "Search the CHORUS knowledge base for relevant documents about research, grants, workshops, and team members.",
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The search query in natural language"
                },
                "top_k": {
                    "type": "integer",
                    "description": "Number of results to return (default 5)",
                    "default": 5
                }
            },
            "required": ["query"]
        }
    },
    {
        "name": "get_context_for_question",
        "description": "Get relevant context from the knowledge base to answer a question. Use this before answering questions about research, grants, workshops, or team members.",
        "input_schema": {
            "type": "object",
            "properties": {
                "question": {
                    "type": "string",
                    "description": "The question to find context for"
                },
                "max_sources": {
                    "type": "integer",
                    "description": "Maximum number of sources to include",
                    "default": 5
                }
            },
            "required": ["question"]
        }
    },
    {
        "name": "list_available_sources",
        "description": "List all source documents available in the knowledge base.",
        "input_schema": {
            "type": "object",
            "properties": {},
            "required": []
        }
    }
]


def handle_tool_call(tool_name: str, tool_input: dict) -> str:
    """Execute a tool call and return the result."""
    if tool_name == "search_documents":
        results = rag_server.search(
            tool_input["query"],
            top_k=tool_input.get("top_k", 5)
        )
        return json.dumps(results, indent=2)
    
    elif tool_name == "get_context_for_question":
        context, citations = rag_server.get_context(
            tool_input["question"],
            top_k=tool_input.get("max_sources", 5)
        )
        return json.dumps({
            'context': context,
            'citations': citations
        }, indent=2)
    
    elif tool_name == "list_available_sources":
        sources = rag_server.list_sources()
        return json.dumps({
            'total_sources': len(sources),
            'sources': sources[:30]
        }, indent=2)
    
    return json.dumps({"error": f"Unknown tool: {tool_name}"})


print("Tool definitions and handler ready!")

Tool definitions and handler ready!


In [160]:
def chat_with_rag(user_message: str, api_key: str = None, model: str = "claude-sonnet-4-20250514") -> str:
    """
    Chat with Claude using RAG tools.
    
    Args:
        user_message: The user's question or message
        api_key: Anthropic API key (or set ANTHROPIC_API_KEY env var)
        model: Claude model to use
    
    Returns:
        Claude's response
    """
    client = anthropic.Anthropic(api_key=api_key or os.environ.get("ANTHROPIC_API_KEY"))
    
    messages = [{"role": "user", "content": user_message}]
    
    # System prompt
    system = """You are a helpful assistant with access to the CHORUS knowledge base, 
which contains documents about research projects, grants, workshops, and team members 
at the Knowledge Lab (University of Chicago).

When answering questions about the knowledge base:
1. Use the get_context_for_question tool to retrieve relevant information
2. Base your answers on the retrieved context
3. Cite your sources using the provided citations
4. If the context doesn't contain enough information, say so

Always search the knowledge base before answering questions about research, grants, 
workshops, team members, or any organizational information."""
    
    # Initial API call
    response = client.messages.create(
        model=model,
        max_tokens=4096,
        system=system,
        tools=TOOLS,
        messages=messages
    )
    
    # Handle tool use loop
    while response.stop_reason == "tool_use":
        # Process tool calls
        tool_results = []
        assistant_content = response.content
        
        for block in response.content:
            if block.type == "tool_use":
                print(f"  [Tool: {block.name}]")
                result = handle_tool_call(block.name, block.input)
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": result
                })
        
        # Continue conversation with tool results
        messages.append({"role": "assistant", "content": assistant_content})
        messages.append({"role": "user", "content": tool_results})
        
        response = client.messages.create(
            model=model,
            max_tokens=4096,
            system=system,
            tools=TOOLS,
            messages=messages
        )
    
    # Extract final text response
    final_response = ""
    for block in response.content:
        if hasattr(block, 'text'):
            final_response += block.text
    
    return final_response


print("chat_with_rag() function ready!")
print("\nUsage:")
print('  response = chat_with_rag("What research workshops have been held?")')

chat_with_rag() function ready!

Usage:
  response = chat_with_rag("What research workshops have been held?")


In [ ]:
# Demo: Chat with RAG (requires ANTHROPIC_API_KEY)
# Uncomment to test:

# response = chat_with_rag("What are the main research projects at Knowledge Lab?")
# print(response)

### Option 2: MCP Server for Claude Desktop

Run the MCP server as a standalone process that Claude Desktop can connect to.

In [161]:
# Save the MCP server as a standalone script
mcp_server_code = '''
#!/usr/bin/env python3
"""CHORUS RAG MCP Server

Run this script to start the MCP server for Claude Desktop.
"""

import json
import pickle
import string
from pathlib import Path
from dataclasses import dataclass
from typing import Optional

import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
from rank_bm25 import BM25Okapi
from fastmcp import FastMCP


@dataclass
class ChunkWithMetadata:
    text: str
    source_file: str = ""
    source_path: str = ""
    element_type: str = ""
    chunk_index: int = 0
    total_chunks: int = 0
    page_number: Optional[int] = None
    section: str = ""
    char_start: int = 0
    char_end: int = 0
    
    @classmethod
    def from_dict(cls, d: dict) -> "ChunkWithMetadata":
        return cls(**d)
    
    def citation(self) -> str:
        parts = [self.source_file] if self.source_file else []
        if self.page_number:
            parts.append(f"p.{self.page_number}")
        return " | ".join(parts) if parts else "Unknown"


class QueryExpander:
    ACRONYMS = {
        "ai": "artificial intelligence", "ml": "machine learning",
        "nlp": "natural language processing", "llm": "large language model",
        "nsf": "national science foundation", "nih": "national institutes of health",
    }
    
    def expand_query(self, query: str) -> list[str]:
        queries = [query]
        q_lower = query.lower()
        for acr, full in self.ACRONYMS.items():
            if acr in q_lower.split():
                queries.append(q_lower.replace(acr, full))
        return queries


class RAGServer:
    def __init__(self, index_path: str):
        self.index_path = Path(index_path)
        self.faiss_index = None
        self.bm25_index = None
        self.chunks = []
        self.tokenized_chunks = []
        self.model = None
        self.expander = QueryExpander()
        
    def load(self):
        self.faiss_index = faiss.read_index(str(self.index_path / "faiss_index.faiss"))
        with open(self.index_path / "hybrid_data.pkl", "rb") as f:
            data = pickle.load(f)
            self.chunks = [ChunkWithMetadata.from_dict(c) for c in data["chunks"]]
            self.tokenized_chunks = data["tokenized_chunks"]
        self.bm25_index = BM25Okapi(self.tokenized_chunks)
        self.model = SentenceTransformer("all-MiniLM-L6-v2")
        
    def search(self, query: str, top_k: int = 5) -> list[dict]:
        queries = self.expander.expand_query(query)
        results = {}
        
        for i, q in enumerate(queries):
            weight = 1.0 if i == 0 else 0.5
            emb = self.model.encode([q], convert_to_numpy=True)
            faiss.normalize_L2(emb)
            scores, indices = self.faiss_index.search(emb, top_k * 2)
            
            tokens = [t for t in q.lower().translate(str.maketrans("", "", string.punctuation)).split() if len(t) > 2]
            kw_scores = self.bm25_index.get_scores(tokens)
            max_kw = max(kw_scores) if max(kw_scores) > 0 else 1
            
            for idx, sem in zip(indices[0], scores[0]):
                if idx < 0: continue
                combined = 0.7 * sem + 0.3 * (kw_scores[idx] / max_kw)
                if idx not in results:
                    results[idx] = {"text": self.chunks[idx].text, "citation": self.chunks[idx].citation(), "score": 0}
                results[idx]["score"] += combined * weight
        
        return sorted(results.values(), key=lambda x: x["score"], reverse=True)[:top_k]


# Initialize
rag = RAGServer("Data/rag_indexes/hybrid")
rag.load()

mcp = FastMCP("CHORUS RAG")

@mcp.tool()
def search_documents(query: str, top_k: int = 5) -> str:
    """Search the CHORUS knowledge base."""
    return json.dumps(rag.search(query, top_k), indent=2)

@mcp.tool()
def get_context(question: str, max_sources: int = 5) -> str:
    """Get context to answer a question about research, grants, or team."""
    results = rag.search(question, max_sources)
    context = "\\n\\n".join(f"[{r[\'citation\']}]\\n{r[\'text\']}" for r in results)
    return json.dumps({"context": context, "citations": [r["citation"] for r in results]})

@mcp.tool()
def list_sources() -> str:
    """List available source documents."""
    sources = {}
    for c in rag.chunks:
        if c.source_file not in sources:
            sources[c.source_file] = 0
        sources[c.source_file] += 1
    return json.dumps(sorted([{"file": k, "chunks": v} for k, v in sources.items()], key=lambda x: -x["chunks"])[:30])

if __name__ == "__main__":
    mcp.run()
'''

with open("mcp_rag_server.py", "w") as f:
    f.write(mcp_server_code)

print("Saved mcp_rag_server.py")
print("\nTo use with Claude Desktop, add to your claude_desktop_config.json:")
print('''
{
  "mcpServers": {
    "chorus-rag": {
      "command": "python",
      "args": ["/path/to/chorus/mcp_rag_server.py"]
    }
  }
}
''')

Saved mcp_rag_server.py

To use with Claude Desktop, add to your claude_desktop_config.json:

{
  "mcpServers": {
    "chorus-rag": {
      "command": "python",
      "args": ["/path/to/chorus/mcp_rag_server.py"]
    }
  }
}



## Summary

This notebook provides two integration options:

### 1. Direct API Integration (Recommended)
```python
response = chat_with_rag("What research workshops have been held?")
```
- Uses Anthropic API with tool definitions
- Claude automatically searches the knowledge base
- Returns answers with citations

### 2. MCP Server for Claude Desktop
```bash
python mcp_rag_server.py
```
- Runs as standalone MCP server
- Connect from Claude Desktop
- Tools available in chat interface

### Available Tools
| Tool | Description |
|------|-------------|
| `search_documents` | Search knowledge base with hybrid retrieval |
| `get_context_for_question` | Get formatted context with citations |
| `list_available_sources` | List all indexed documents |
| `search_by_source` | Search within a specific document |